
Kết nối drive




In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Cài đặt thư viện

In [2]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes sentencepiece evaluate scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.9 MB/s eta 0:00:00


Import thư viện

In [3]:
import os
import json
import torch
import random

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from huggingface_hub import login

print("Torch Version :", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



Torch Version : 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4


In [4]:
ROOT_DRIVE = "/content/drive/MyDrive/gemmapoem"
TRAIN_FILE = f"{ROOT_DRIVE}/data/train.txt"
TEST_FILE  = f"{ROOT_DRIVE}/data/test.txt"
CHECKPOINT_DIR = f"{ROOT_DRIVE}/checkpoints"
LORA_MODEL = f"{ROOT_DRIVE}models/gemma2b-lucbat-lora"
MERGED_MODEL = f"{ROOT_DRIVE}/models/gemma2b-lucbat"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [8]:
from huggingface_hub import login, whoami
from google.colab import userdata

try:
    login(token=userdata.get("HF_TOKEN"))
    print(whoami())
except Exception as e:
    print("Vui lòng thiết lập HF_TOKEN trong Colab Secrets.")

{'type': 'user', 'id': '6a59f7403262e70992599fb5', 'name': 'NguyenThiAnhThu', 'fullname': 'NguyenThiAnhThu', 'email': 'nguyenthutcmh@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1785542400, 'isPro': False, 'avatarUrl': '/avatars/517cc95fd098719679fbd239d12b40ee.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'GemmaColab', 'role': 'read', 'createdAt': '2026-07-25T18:06:32.250Z'}}}


Load tokenizer, model

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name="google/gemma-2-2b"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Cấu hình nén 4-bit
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load model 4bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    attn_implementation="eager"
)

# Bắt buộc khi dùng Gradient Checkpointing trong TrainingArguments
model.config.use_cache = False
model.gradient_checkpointing_enable()

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Test model

In [10]:
import torch

# 1. Mồi trước một vài câu thơ để mô hình "bắt chước" viết tiếp (Mô hình Base học rất nhanh qua các ví dụ mồi - Few-shot prompting)
prompt = """Thơ lục bát Việt Nam:

Câu 1:
Trăm năm trong cõi người ta,
Chữ tài chữ mệnh khéo là ghét nhau.

Câu 2:
Thân em như chẽn lúa đòng đòng,
Phất phơ dưới ngọn nắng hồng ban mai.

Câu 3:
"""

# 2. Mã hóa văn bản thành Tokens (Bản Base KHÔNG dùng apply_chat_template)
model_inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

# 3. Tiến hành sinh văn bản (Text Generation)
with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=100,        # Giới hạn độ dài sinh thêm
        do_sample=True,            # Bật lấy mẫu ngẫu nhiên để tăng tính sáng tạo
        temperature=0.8,           # Độ sáng tạo (Thơ ca nên để từ 0.7 - 0.9)
        top_k=40,                  # Chỉ chọn từ trong top 40 từ có xác suất cao nhất
        top_p=0.9,                 # Giới hạn phân phối xác suất từ vựng
        repetition_penalty=1.2,    # Hình phạt lặp từ (Rất quan trọng cho bản Base để tránh bị lặp)
        pad_token_id=tokenizer.eos_token_id
    )

# 4. Giải mã kết quả đầu ra
response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("\n--- KẾT QUẢ TEST MÔ HÌNH BASE (TRƯỚC KHI TRAIN) ---")
print(response)


--- KẾT QUẢ TEST MÔ HÌNH BASE (TRƯỚC KHI TRAIN) ---
Thơ lục bát Việt Nam:

Câu 1:
Trăm năm trong cõi người ta,
Chữ tài chữ mệnh khéo là ghét nhau.

Câu 2:
Thân em như chẽn lúa đòng đòng,
Phất phơ dưới ngọn nắng hồng ban mai.

Câu 3:
Sách cũ nói tình hay thật thà!
Bởi đôi môi không muốn mình trả lời mà thôi.

Câu 4:
Mối duyên này ai đã hứa?
Lỡ một ngày chẳng còn thì làm sao nào?

Ai giúp e với ạ cảm ơn nhìu!!!


Dùng từ láy và biện pháp tu từ đặc sắc để tả cảnh mùa xuân ở nhà em.(ko copy mạng nha)



Bài 7 (5 điểm): Câu


Prepare dataset

In [11]:
import os
import json

TRAIN_JSON = TRAIN_FILE.replace(".txt", ".json")
TEST_JSON = TEST_FILE.replace(".txt", ".json")

def process_file(txt_path, json_path, label):
    with open(txt_path, "r", encoding="utf-8") as f:
        # Đọc toàn bộ file và tách thành danh sách các bài thơ dựa trên 2 dấu xuống dòng
        raw_content = f.read().strip()
        poems = [p.strip() for p in raw_content.split("\n\n") if p.strip()]

    # Gom tất cả các bài thơ thành một danh sách các dictionary
    data_list = [{"text": poem} for poem in poems]

    # Ghi toàn bộ danh sách vào file JSON
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data_list, f, ensure_ascii=False, indent=4)

    print(f"Đã chuyển đổi tập {label.upper()} thành công!")
    print(f"File JSON mới: {json_path} (Tổng số bài: {len(data_list)})")

process_file(TRAIN_FILE, TRAIN_JSON, "train")
process_file(TEST_FILE, TEST_JSON, "test")

Đã chuyển đổi tập TRAIN thành công!
File JSON mới: /content/drive/MyDrive/gemmapoem/data/train.json (Tổng số bài: 67754)
Đã chuyển đổi tập TEST thành công!
File JSON mới: /content/drive/MyDrive/gemmapoem/data/test.json (Tổng số bài: 38014)


Load dataset

In [12]:
from datasets import load_dataset

# Tập train, test
dataset=load_dataset("json", data_files={"train": TRAIN_JSON, "test": TEST_JSON,},)

# Tập eval
num_eval = min(1000, len(dataset["test"]))
eval = dataset["test"].select(range(num_eval))

# Tập debug: small_train, small_eval
num_train_samples = min(100, len(dataset["train"]))
num_eval_samples = min(20, len(dataset["test"]))
small_train = dataset["train"].select(range(num_train_samples))
small_eval = dataset["test"].select(range(num_eval_samples))

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Packing (Constant Length Dataset)


* Gộp nhiều bài thơ ngắn thành một chuỗi token liên tục.
* Chèn token <eos> giữa các bài thơ để đánh dấu điểm kết thúc.
* Cắt chuỗi dài thành các khối có độ dài cố định (block_size = 512).



In [13]:
block_size = 512
def group_texts(examples):

    # Thêm token kết thúc (<eos>) vào cuối mỗi bài thơ
    texts = [text + tokenizer.eos_token for text in examples["text"]]

    # Tokenize toàn bộ các bài thơ
    tokenized = tokenizer(texts, add_special_tokens=False,)

    # Ghép toàn bộ danh sách token của nhiều bài thơ thành một chuỗi token liên tục
    concatenated = []
    for ids in tokenized["input_ids"]: concatenated.extend(ids)

    # Bỏ phần token dư cuối cùng nếu không đủ tạo thành một block hoàn chỉnh
    total_length = (len(concatenated) // block_size) * block_size
    concatenated = concatenated[:total_length]

    # Chia chuỗi token dài thành nhiều block có cùng kích thước block_size
    result = {"input_ids":[concatenated[i:i+block_size] for i in range(0, total_length, block_size)]}
    result["labels"] = result["input_ids"].copy()

    return result

# Áp dụng Packing cho toàn bộ tập Train
packed_train = dataset["train"].map(
    group_texts,
    batched=True,                           # Xử lý nhiều mẫu cùng lúc để tăng tốc
    remove_columns=["text"],                # Xóa cột văn bản gốc sau khi tokenize
    desc="Packing train dataset"            # Hiển thị tiến trình xử lý
)

# Áp dụng Packing cho tập Validation
packed_eval = eval.map(group_texts, batched=True, remove_columns=["text"], desc="Packing eval dataset")

# Áp dụng Packing cho tập Debug
packed_small_eval = small_eval.map(group_texts, batched=True, remove_columns=["text"], desc="Packing eval dataset")
packed_small_train = small_train.map(group_texts, batched=True, remove_columns=["text"], desc="Packing eval dataset")

Packing train dataset:   0%|          | 0/67754 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Data Collator

In [14]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

LoRA

In [15]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# Chuẩn bị model cho k-bit training
model = prepare_model_for_kbit_training(model)

#Tạo LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# Gắn LoRA
model = get_peft_model(model, lora_config)

# Hiển thị số lượng tham số được huấn luyện (LoRA) và tổng số tham số của mô hình
model.print_trainable_parameters()

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881


Training Arguments

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir=CHECKPOINT_DIR,  # Thư mục lưu checkpoint (nên trỏ thẳng lên Google Drive)
    dataloader_num_workers=2,   # Sử dụng 2 luồng CPU nạp dữ liệu song song, giữ cho GPU không bị "đói" dữ liệu

    fp16=True,               # Tắt FP16 vì dễ gây xung đột kiểu dữ liệu
    bf16=False,                # Bật BF16 thuần để đồng bộ hoàn hảo với định dạng gốc
    optim="paged_adamw_8bit", # Bộ tối ưu hóa lượng hóa 8-bit, tiết kiệm 75% VRAM giúp chống tràn bộ nhớ trên T4
    report_to="none",         # Tắt đồng bộ bên thứ ba (WandB, Tensorboard) để giảm độ trễ mạng và nhẹ code

    num_train_epochs=2,             # Số lần mô hình học lặp lại toàn bộ tập dữ liệu (2 vòng là vừa đủ)
    per_device_train_batch_size=2, # Số lượng mẫu từ tập Train được đẩy vào GPU xử lý cùng một lúc
    per_device_eval_batch_size=2,   # Số lượng mẫu từ tập Test/Valid được xử lý cùng một lúc khi làm bài kiểm tra
    gradient_accumulation_steps=8,  # Cập nhật trọng số ngay sau mỗi batch để đạt tốc độ lặp (it/s) nhanh nhất

    gradient_checkpointing=True,

    learning_rate=2e-4,         # Tốc độ học tối ưu khi áp dụng LoRA cho các mô hình ngôn ngữ nhỏ 1B-3B
    lr_scheduler_type="cosine", # Giảm tốc độ học dần về cuối theo đường cong Cosine để dò điểm tối ưu chính xác
    warmup_steps=100,           # Tăng dần tốc độ học trong 100 bước đầu tiên để mô hình không bị "sốc" dữ liệu mới
    logging_steps=50,          # Cứ sau 50 bước huấn luyện sẽ in chỉ số Loss (mức độ sai số) ra màn hình để theo dõi

    eval_strategy="steps",  # Bật tính năng tự động đánh giá định kỳ dựa trên số bước
    eval_steps=50,         # Cứ sau 50 steps thì chạy đánh giá tập test một lần

    save_strategy="steps",  # Bật tính năng tự động lưu checkpoint định kỳ dựa trên số bước (steps)
    save_steps=50,         # Cứ sau 50 bước lặp thì lưu checkpoint một lần (Bắt buộc trùng với eval_steps)
    save_total_limit=1,     # Giới hạn số lượng checkpoint tối đa được lưu trên ổ cứng (GG Drive) tại một thời điểm

    metric_for_best_model="eval_loss",  # Chọn điểm lỗi trên tập test làm tiêu chí đánh giá mô hình hay/dở
    greater_is_better=False,            # Điểm lỗi (Loss) càng thấp (nhỏ) chứng tỏ mô hình làm thơ càng chuẩn luật
    load_best_model_at_end=True,        # Tự động tải lại checkpoint có loss thấp nhất đè lên mô hình khi kết thúc train
)

Trainer

In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    # Sử dụng dữ liệu đã Packing
    train_dataset=packed_train,
    eval_dataset=packed_eval,

    processing_class=tokenizer,
    data_collator=data_collator,
)

Train

In [18]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if last_checkpoint is not None:
    print(f"Resume từ {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Train từ đầu.")
    trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Resume từ /content/drive/MyDrive/gemmapoem/checkpoints/checkpoint-2550


Step,Training Loss,Validation Loss
2576,2.321328,2.869998


[transformers] Could not locate the best model at /content/drive/MyDrive/gemmapoem/checkpoints/checkpoint-2250/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/gemmapoem/checkpoints/checkpoint-2250'

Lưu LoRA Adapter

In [ ]:
trainer.model.save_pretrained(LORA_MODEL)
tokenizer.save_pretrained(LORA_MODEL)

('/content/drive/MyDrive/gen-poemmodels/llama32-lucbat-lora/tokenizer_config.json',
 '/content/drive/MyDrive/gen-poemmodels/llama32-lucbat-lora/tokenizer.json')

In [ ]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Merge LoRA Adapter + Base Model

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(LORA_MODEL)

# Load LoRA Adapter
model = AutoPeftModelForCausalLM.from_pretrained(
    LORA_MODEL,
    dtype=torch.float16,
    device_map="auto"
)

# Merge LoRA Adapter vào Base model
merged_model = model.merge_and_unload()

# Lưu mô hình hoàn chỉnh
merged_model.save_pretrained(MERGED_MODEL)
tokenizer.save_pretrained(MERGED_MODEL)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/gen-poem/models/llama32-lucbat/tokenizer_config.json',
 '/content/drive/MyDrive/gen-poem/models/llama32-lucbat/tokenizer.json')

Test model sau khi train

In [ ]:
import torch

# 1. Mồi trước một vài câu thơ để mô hình "bắt chước" viết tiếp (Mô hình Base học rất nhanh qua các ví dụ mồi - Few-shot prompting)
prompt = """Thơ lục bát Việt Nam:

Câu 1:
Trăm năm trong cõi người ta,
Chữ tài chữ mệnh khéo là ghét nhau.

Câu 2:
Thân em như chẽn lúa đòng đòng,
Phất phơ dưới ngọn nắng hồng ban mai.

Câu 3:
"""

# prompt = "Viết một bài thơ lục bát về mùa thu."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Thơ lục bát Việt Nam:

Câu 1:
Trăm năm trong cõi người ta,
Chữ tài chữ mệnh khéo là ghét nhau.

Câu 2:
Thân em như chẽn lúa đòng đòng,
Phất phơ dưới ngọn nắng hồng ban mai.

Câu 3:
Mấy mươi tuổi, hai con đường đi,
Dựng nhà bằng lòng thương nhớ ai.

Câu 4:
Tháng tháng năm bảy ba bốn năm mười,
Đôi đôi mắt hạnh phúc hẹn gặp nhau.

Câu 5:
Con rắn vàng tràn cả sông hồ nước,
Lối gió sấm sét nổ lên tiếng cười vang.
Đó là con thánh thiện, con lành anh hùng!
Giúp đỡ cho người khi cần may mắn.

Câu 6:
Nước xa xôi, tuyết m


Nén và tải về máy

In [ ]:
ZIP_FILE = f"{ROOT_DRIVE}/models/llama32-lucbat.zip"
!zip -r "{ZIP_FILE}" "{MERGED_MODEL}"

from google.colab import files
files.download(ZIP_FILE)

  adding: content/drive/MyDrive/gen-poem/models/llama32-lucbat/ (stored 0%)
  adding: content/drive/MyDrive/gen-poem/models/llama32-lucbat/config.json (deflated 52%)
  adding: content/drive/MyDrive/gen-poem/models/llama32-lucbat/generation_config.json (deflated 32%)
  adding: content/drive/MyDrive/gen-poem/models/llama32-lucbat/model.safetensors (deflated 13%)
  adding: content/drive/MyDrive/gen-poem/models/llama32-lucbat/tokenizer_config.json (deflated 49%)
  adding: content/drive/MyDrive/gen-poem/models/llama32-lucbat/tokenizer.json (deflated 85%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>